# 01 · Retrieve and process BTC/USDT 1-second klines (Jan 2023 onwards)

**Goal.** Retrieve every 1-second BTC/USDT spot kline since January 2023 from Binance's public data, check its quality, clean it, and aggregate it into 1-minute bars that keep the information only 1-second data can provide.

**Source.** [Binance Public Data](https://data.binance.vision), the official bulk archive of the `/api/v3/klines` endpoint, one zip per month, free and with no API key. Bulk monthly files are far more efficient than the REST endpoint, which returns at most 1,000 klines (≈17 minutes of 1-second data) per request.

**Scale.** ~2.7 million rows per month, ~120 million rows in total. To keep memory and disk use small, each month is downloaded, processed in memory and reduced to 1-minute bars before moving on. The raw 1-second data for January 2023 is kept as a sample.

**Pipeline**
1. Setup
2. Download one month and verify its SHA-256 checksum
3. Walk through the cleaning steps on January 2023 (steps 4–10)
4. Wrap the steps in a function
5. Loop over all months and save the results

## Step 1 · Imports

In [ ]:
import os, time, hashlib
import requests, zipfile, io
import numpy as np
import pandas as pd

## Step 2 · Settings

Binance's 1-second kline files have no header row, so columns are named following the official documentation.

| column | meaning |
|---|---|
| open_time | start of the second |
| open / high / low / close | prices in USDT |
| volume | traded volume in BTC |
| close_time | end of the second |
| quote_volume | traded value in USDT |
| trades | number of trades |
| taker_buy_base | BTC bought by aggressive buyers (takers) |
| taker_buy_quote | USDT value bought by takers |
| ignore | unused |

In [ ]:
OUT_DIR = "data_1min"
os.makedirs(OUT_DIR, exist_ok=True)

COLS = ["open_time", "open", "high", "low", "close", "volume", "close_time",
        "quote_volume", "trades", "taker_buy_base", "taker_buy_quote", "ignore"]
BASE_URL = "https://data.binance.vision/data/spot/monthly/klines/BTCUSDT/1s/"

START, END = "2023-01", "2026-10"   # full months covered

## Step 3 · Download one month and verify it

Each monthly zip has a published SHA-256 checksum (`.CHECKSUM` file). The hash of the downloaded bytes is compared with it to confirm the file is complete and uncorrupted. January has 31 × 86,400 = 2,678,400 seconds.

In [ ]:
def download_month(month):
    url = f"{BASE_URL}BTCUSDT-1s-{month}.zip"
    r = requests.get(url, timeout=900)
    if r.status_code != 200:
        print(month, "download failed, status", r.status_code)
        return None, False
    official = requests.get(url + ".CHECKSUM", timeout=60).text.split()[0]
    checksum_ok = hashlib.sha256(r.content).hexdigest() == official
    z = zipfile.ZipFile(io.BytesIO(r.content))
    raw = pd.read_csv(z.open(z.namelist()[0]), header=None, names=COLS)
    return raw, checksum_ok

raw, ok = download_month("2023-01")
print("checksum OK:", ok)
print("rows:", f"{len(raw):,}", "| seconds in month:", f"{31 * 86400:,}")
raw.head()

## Step 4 · Standardise timestamps

`open_time` counts time units since 1 Jan 1970. Binance switched spot timestamps from **milliseconds** to **microseconds** on 1 Jan 2025, so the unit is detected from the magnitude (values above $10^{14}$ are microseconds). Treating microseconds as milliseconds would produce dates tens of thousands of years in the future.

In [ ]:
df = raw.copy()
unit = "us" if df["open_time"].iloc[0] > 1e14 else "ms"
df["time"] = pd.to_datetime(df["open_time"], unit=unit)
print("unit:", unit, "| first:", df["time"].min(), "| last:", df["time"].max())

## Step 5 · Duplicates

Any second recorded twice is counted and removed.

In [ ]:
n_dup = int(df["time"].duplicated().sum())
df = df.drop_duplicates("time").set_index("time").sort_index()
print("duplicate rows:", n_dup)

## Step 6 · Integrity checks (flag, do not delete)

A valid kline satisfies $\text{low} \le \text{open}, \text{close} \le \text{high}$ and $\text{volume} \ge 0$. Violations are counted for review. Very large volumes are genuine market events (block trades, liquidations, news) and are kept; seconds with volume above 50× the trailing 1-hour median are counted to describe how often they occur.

In [ ]:
bad = ((df["high"] < df["low"]) |
       (df["open"] > df["high"]) | (df["open"] < df["low"]) |
       (df["close"] > df["high"]) | (df["close"] < df["low"]) |
       (df["volume"] < 0))
n_bad = int(bad.sum())
n_spike = int((df["volume"] > 50 * df["volume"].rolling(3600, min_periods=600).median()).sum())
print("rows failing OHLC checks:", n_bad, "| volume spikes (>50x 1h median):", n_spike)

## Step 7 · Fill missing seconds

Seconds with no trades (or exchange maintenance) are absent from the file. The data is re-indexed onto a complete 1-second grid: the price is carried forward from the previous close (no trade means no price change) and volume, trades and taker volume are set to zero. Dropping these seconds would break the time axis; averaging would invent prices.

In [ ]:
full = pd.date_range(df.index[0].floor("D"), df.index[-1].ceil("D") - pd.Timedelta("1s"), freq="1s")
n_missing = len(full) - len(df)
df = df.reindex(full)
df["close"] = df["close"].ffill().bfill()
for c in ["open", "high", "low"]:
    df[c] = df[c].fillna(df["close"])
for c in ["volume", "quote_volume", "taker_buy_base", "trades"]:
    df[c] = df[c].fillna(0)
zero_share = float((df["trades"] == 0).mean())
print("seconds:", f"{len(full):,}", "| missing & filled:", n_missing, f"| seconds with no trades: {zero_share:.4%}")

## Step 8 · 1-second log returns

$$r_i = \ln P_i - \ln P_{i-1}$$

Log returns are additive over time, so the 60 one-second returns in a minute sum exactly to that minute's return:
$$\sum_{i=1}^{60} r_i = \ln P_{60} - \ln P_0$$

In [ ]:
lp = np.log(df["close"])
r1 = lp.diff().fillna(0)
r1.head()

## Step 9 · Aggregate to 1-minute bars plus 1-second features

OHLCV aggregation: open = first, high = max, low = min, close = last, volumes and trade counts = sum.

Features that require 1-second data:
- **rv**. realized variance, $RV = \sum_{i=1}^{60} r_i^2$, a precise measure of intra-minute volatility ($\sqrt{RV}$ is the minute's volatility)
- **active_secs**. seconds within the minute with at least one trade (liquidity)
- **max_abs_1s_ret**. largest absolute 1-second move in the minute (jumps)
- **max_1s_volume**. largest 1-second volume in the minute
- **ret_last10s**. log return over the last 10 seconds of the minute

Two reconciliation checks follow: total volume must match between the 1-second and 1-minute data, and the sum of 1-second returns must equal the 1-minute return.

In [ ]:
m = df.resample("1min").agg({"open": "first", "high": "max", "low": "min", "close": "last",
                             "volume": "sum", "quote_volume": "sum",
                             "trades": "sum", "taker_buy_base": "sum"})
m["rv"] = (r1 ** 2).resample("1min").sum()
m["active_secs"] = (df["trades"] > 0).resample("1min").sum()
m["max_abs_1s_ret"] = r1.abs().resample("1min").max()
m["max_1s_volume"] = df["volume"].resample("1min").max()
m["ret_last10s"] = lp.resample("1min").last() - lp.shift(10).resample("1min").last()

print("volume 1s:", round(df["volume"].sum(), 4), "| volume 1min:", round(m["volume"].sum(), 4))
diff = (r1.resample("1min").sum().iloc[1:] - np.log(m["close"]).diff().iloc[1:]).abs().max()
print("max difference, summed 1s returns vs 1min returns:", float(diff))
print("1-minute rows:", f"{len(m):,}")
m.head()

## Step 10 · Monthly second-level statistics

- **autocorr_1s**. lag-1 autocorrelation of 1-second returns. Negative values indicate bid-ask bounce (prices flick back and forth).
- **Variance ratio**
$$VR(k) = \frac{\operatorname{Var}(k\text{-second return})}{k \cdot \operatorname{Var}(1\text{-second return})}$$
$VR = 1$ for a random walk, $VR < 1$ indicates mean reversion, $VR > 1$ indicates trending. Non-overlapping $k$-second returns are used.

In [ ]:
lc = lp.to_numpy()
var_1s = np.var(np.diff(lc))
stats = {"month": "2023-01", "raw_rows": len(raw), "checksum_ok": ok,
         "seconds_in_month": len(full), "missing_seconds": n_missing, "duplicate_rows": n_dup,
         "bad_ohlc_rows": n_bad, "volume_spike_secs_50x": n_spike,
         "zero_trade_share": round(zero_share, 4), "autocorr_1s": round(float(r1.autocorr(1)), 4)}
for k in [10, 60, 300, 3600]:
    rk = (lc[k:] - lc[:-k])[::k]
    stats[f"VR_{k}s"] = round(float(np.var(rk) / (k * var_1s)), 3)
stats

## Step 11 · Wrap steps 4–10 in a function

In [ ]:
for _v in ["raw", "df", "m", "lp", "r1", "lc"]:   # free memory from the walkthrough
    globals().pop(_v, None)

def process_month(raw):
    df = raw.copy()
    unit = "us" if df["open_time"].iloc[0] > 1e14 else "ms"
    df["time"] = pd.to_datetime(df["open_time"], unit=unit)
    n_dup = int(df["time"].duplicated().sum())
    df = df.drop_duplicates("time").set_index("time").sort_index()
    vol_before = df["volume"].sum()
    bad = ((df["high"] < df["low"]) |
           (df["open"] > df["high"]) | (df["open"] < df["low"]) |
           (df["close"] > df["high"]) | (df["close"] < df["low"]) |
           (df["volume"] < 0))
    n_bad = int(bad.sum())
    n_spike = int((df["volume"] > 50 * df["volume"].rolling(3600, min_periods=600).median()).sum())
    full = pd.date_range(df.index[0].floor("D"), df.index[-1].ceil("D") - pd.Timedelta("1s"), freq="1s")
    n_missing = len(full) - len(df)
    df = df.reindex(full)
    df["close"] = df["close"].ffill().bfill()
    for c in ["open", "high", "low"]:
        df[c] = df[c].fillna(df["close"])
    for c in ["volume", "quote_volume", "taker_buy_base", "trades"]:
        df[c] = df[c].fillna(0)
    lp = np.log(df["close"])
    r1 = lp.diff().fillna(0)
    m = df.resample("1min").agg({"open": "first", "high": "max", "low": "min", "close": "last",
                                 "volume": "sum", "quote_volume": "sum",
                                 "trades": "sum", "taker_buy_base": "sum"})
    m["rv"] = (r1 ** 2).resample("1min").sum()
    m["active_secs"] = (df["trades"] > 0).resample("1min").sum()
    m["max_abs_1s_ret"] = r1.abs().resample("1min").max()
    m["max_1s_volume"] = df["volume"].resample("1min").max()
    m["ret_last10s"] = lp.resample("1min").last() - lp.shift(10).resample("1min").last()
    lc = lp.to_numpy()
    var_1s = np.var(np.diff(lc))
    s = {"seconds_in_month": len(full), "missing_seconds": n_missing, "duplicate_rows": n_dup,
         "bad_ohlc_rows": n_bad, "volume_spike_secs_50x": n_spike,
         "zero_trade_share": round(float((df["trades"] == 0).mean()), 4),
         "autocorr_1s": round(float(r1.autocorr(1)), 4),
         "volume_check_ok": bool(np.isclose(m["volume"].sum(), vol_before))}
    for k in [10, 60, 300, 3600]:
        rk = (lc[k:] - lc[:-k])[::k]
        s[f"VR_{k}s"] = round(float(np.var(rk) / (k * var_1s)), 3)
    return m, s

## Step 12 · Process all months

Months already processed are skipped, so the loop can be resumed if interrupted.

In [ ]:
stats_path = f"{OUT_DIR}/monthly_stats.csv"
all_stats = pd.read_csv(stats_path).to_dict("records") if os.path.exists(stats_path) else []

for month in pd.period_range(START, END, freq="M").astype(str):
    path = f"{OUT_DIR}/BTCUSDT-1min-{month}.parquet"
    if os.path.exists(path):
        print(month, "already processed, skipping")
        continue
    t0 = time.time()
    raw, ok = download_month(month)
    if raw is None:
        continue
    if month == "2023-01":
        raw.to_parquet(f"{OUT_DIR}/SAMPLE_BTCUSDT-1s-2023-01.parquet", index=False)
    m, s = process_month(raw)
    m.to_parquet(path)
    s.update({"month": month, "raw_rows": len(raw), "checksum_ok": ok})
    all_stats.append(s)
    pd.DataFrame(all_stats).to_csv(stats_path, index=False)
    print(month, f"{len(raw):,} rows", "checksum OK" if ok else "CHECKSUM MISMATCH",
          f"| missing {s['missing_seconds']}s | bad rows {s['bad_ohlc_rows']} | {time.time() - t0:.0f}s")
    del raw, m
print("done")

## Step 13 · Data-quality summary

In [ ]:
summary = pd.read_csv(stats_path)
cols = ["month", "raw_rows", "checksum_ok", "missing_seconds", "duplicate_rows",
        "bad_ohlc_rows", "zero_trade_share"]
print("total 1-second rows retrieved:", f"{summary['raw_rows'].sum():,}")
print("all checksums OK:", bool(summary["checksum_ok"].all()))
summary[cols]